<a href="https://colab.research.google.com/github/Addreta/Subgraph-Centrality/blob/main/real_networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pennylane

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 9.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 3.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 9.6 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 14.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 13.0 MB/s  0:00:00
  Attempting uninstall: autograd
    Found existing installation: autograd 1.9.1
    Uninstalling autograd-1.9.1:
      Successfully uninstalled autograd-1.9.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [pennylane]


In [5]:
#this cell holds the whole ideal-circuit pipeline as functions, so the same steps can be run on any graph.
#the steps : exact SC (numpy) -> normalization and Chebyshev polynomials -> the two ideal QSVT circuits -> ranking comparison.
#bipartite graphs have no odd closed walks, so for them only the even circuit is run
import numpy as np, networkx as nx, pandas as pd, time, warnings
import pennylane as qml
from scipy.linalg import expm
from scipy.special import ive
from scipy.stats import spearmanr, kendalltau
from numpy.polynomial import chebyshev as C
warnings.filterwarnings("ignore")

def tie_classes(sc, tol=1e-9):                    # vertices whose SC agree to 1e-9 (relative) are one class
    order = np.argsort(-sc)
    cls = np.zeros(len(sc), dtype=int)
    for a, b in zip(order[:-1], order[1:]):
        cls[b] = cls[a] + int(abs(sc[a] - sc[b]) > tol * abs(sc[a]))
    return cls                                    # class 0 = most central

def cheb_coeffs(z, deg, parity):
    k = np.arange(deg + 1)
    c = 2.0 * ive(k, z)
    c[0] = ive(0, z)
    c[k % 2 != parity] = 0.0
    if parity == 0:
        return c / (0.5 * (1 + np.exp(-2 * z)))    # divide by cosh(z), scaled by e^-z to match ive
    return c / (0.5 * (1 - np.exp(-2 * z)))        # divide by sinh(z), scaled by e^-z to match ive

def tail_error(z, deg, parity):
    return float(np.sum(np.abs(cheb_coeffs(z, deg + 60, parity)[deg + 1:])))

def degree_for_eps(z, eps, parity, max_deg=300):
    d = parity
    while tail_error(z, d, parity) > eps:
        d += 2
        if d > max_deg:
            raise RuntimeError("tolerance not reached; raise max_deg")
    return d, cheb_coeffs(z, d, parity)

def compare(sc_exact, sc_est, k):
    cls = tie_classes(sc_exact)
    if cls.max() == 0:
        return dict(rho=np.nan, tau=np.nan, overlap=np.nan)
    est = np.array(sc_est, dtype=float); ex = np.array(sc_exact, dtype=float)
    for c in np.unique(cls):
        est[cls == c] = est[cls == c].mean()      # same value inside a tie class
        ex[cls == c] = ex[cls == c].mean()
    top_ex = set(np.argsort(-ex, kind="stable")[:k]); top_est = set(np.argsort(-est, kind="stable")[:k])
    return dict(rho=spearmanr(ex, est)[0], tau=kendalltau(ex, est)[0], overlap=len(top_ex & top_est) / k)

def run_graph(G, beta=1.0):
    t0 = time.time()
    # ---- exact reference (numpy) and the networkx check ----
    nodes = sorted(G.nodes()); n = len(nodes)
    q = int(np.ceil(np.log2(n))); N = 2 ** q                       # padding: add isolated dummy vertices up to 2^q
    A = nx.to_numpy_array(G, nodelist=nodes, weight=None)
    A_p = np.zeros((N, N)); A_p[:n, :n] = A
    SC_exact = np.diag(expm(beta * A))
    nx_diff = np.nan
    if beta == 1.0:
        sc_nx = nx.subgraph_centrality(G)
        nx_diff = max(abs(sc_nx[v] - SC_exact[i]) for i, v in enumerate(nodes))
    bip = nx.is_bipartite(G)
    # ---- normalization and polynomials ----
    lam = np.max(np.abs(np.linalg.eigvalsh(A)))
    alpha = 1.001 * max(lam, np.sqrt(np.max(np.abs(A @ A).sum(axis=1))))
    z = beta * alpha
    s = np.sort(SC_exact)[::-1]
    gaps = np.abs(np.diff(s)); gaps = gaps[gaps > 1e-9 * s[:-1]]
    abs_gap = gaps.min() if len(gaps) else 0.01 * SC_exact.mean()  # all SC equal: no gap exists, so aim for 1% accuracy
    d_e, c_e = degree_for_eps(z, 0.25 * abs_gap / np.cosh(z), 0)
    d_o, c_o = (None, None) if bip else degree_for_eps(z, 0.25 * abs_gap / np.sinh(z), 1)
    # ---- the ideal circuits ----
    At = A_p / alpha
    be_wires = range(q + 1); rp = q + 1
    BE = qml.BlockEncode(At, wires=be_wires)
    dev = qml.device("default.qubit", wires=q + 2)
    def make_circuit(c):
        angles = qml.poly_to_angles(C.cheb2poly(c), "QSVT")
        pos = [qml.PCPhase(a,  dim=N, wires=be_wires) for a in angles]
        neg = [qml.PCPhase(-a, dim=N, wires=be_wires) for a in angles]
        @qml.qnode(dev)
        def circ(v):
            qml.BasisState(np.array([int(b) for b in format(v, f"0{q}b")]), wires=range(1, 1 + q))
            qml.Hadamard(wires=rp)
            qml.ctrl(qml.QSVT(BE, pos), control=rp, control_values=[0])
            qml.ctrl(qml.QSVT(BE, neg), control=rp, control_values=[1])
            qml.Hadamard(wires=rp)
            return qml.probs(wires=range(q + 2))
        return circ
    def p_success(circ, v):
        bits = [0] + [int(b) for b in format(v, f"0{q}b")] + [0]
        return circ(v)[int("".join(map(str, bits)), 2)]
    circ_e = make_circuit(c_e)
    p_e = np.array([p_success(circ_e, v) for v in range(n)])
    if bip:
        p_o = np.zeros(n)
    else:
        circ_o = make_circuit(c_o)
        p_o = np.array([p_success(circ_o, v) for v in range(n)])
    SC_qsvt = np.cosh(z) * np.sqrt(p_e) + np.sinh(z) * np.sqrt(p_o)
    # ---- comparison ----
    k = 3 if n <= 8 else 5
    m = compare(SC_exact, SC_qsvt, k)
    return dict(n=n, edges=G.number_of_edges(), qubits=q + 2, bipartite=bip, classes=int(tie_classes(SC_exact).max() + 1),
                alpha=alpha, deg_even=d_e, deg_odd=d_o, pe_min=p_e.min(), po_min=(np.nan if bip else p_o.min()),
                nx_diff=nx_diff, MAE=np.mean(np.abs(SC_qsvt - SC_exact)), max_rel_err=np.max(np.abs(SC_qsvt - SC_exact) / SC_exact),
                rho=m["rho"], tau=m["tau"], topk=k, overlap=m["overlap"], seconds=time.time() - t0)

In [8]:
#this cell lists some of the graphs of the Akrobotu et al. paper (Table 1) and runs the pipeline on every one with beta = 1 (ideal circuits, no noise, no shots).
#BA(50,5,7) is built with seed=7; if the paper used another networkx version its random graph can differ, but it has the same size.
#rho = 1.000 is Spearman’s rank correlation between the exact SC ranking and the QSVT ranking. It’s 1 when the two rankings are identical, 0 when unrelated and -1 when reversed.
#tau = 1.000 is Kendall’s tau. It checks every pair of vertices and asks whether both rankings order the pair the same way. It’s 1 when all pairs agree.
#max rel err = 1.1e-02 is the largest relative error in the SC values themselves, over all vertices.it is |QSVT value − exact value| / exact value, so 1.1×10⁻² means the worst vertex is off by 1.1%.
graphs = {
    "Bull":            nx.bull_graph(),
    "K3,5":            nx.complete_bipartite_graph(3, 5),
    "K5":              nx.complete_graph(5),
    "Sedgewick-Maze":  nx.sedgewick_maze_graph(),
    "Florentine":      nx.florentine_families_graph(),
    "Lollipop L10,20": nx.lollipop_graph(10, 20),
    "Davis women":     nx.davis_southern_women_graph(),
    "Karate club":     nx.karate_club_graph(),
    "Tutte":           nx.tutte_graph(),
    "BA(50,5,7)":      nx.barabasi_albert_graph(50, 5, seed=7),

}
rows = {}
for name, G in graphs.items():
    if G is None:
        print(f"{name:<16} skipped (no edge list yet)")
        continue
    rows[name] = run_graph(G, beta=1.0)
    r = rows[name]
    print(f"{name:<16} n = {r['n']:>2}  qubits = {r['qubits']}  rho = {r['rho']:.3f}  tau = {r['tau']:.3f}  max rel err = {r['max_rel_err']:.1e}  ({r['seconds']:.0f} s)")

Bull             n =  5  qubits = 5  rho = 1.000  tau = 1.000  max rel err = 1.1e-02  (0 s)
K3,5             n =  8  qubits = 5  rho = 1.000  tau = 1.000  max rel err = 2.4e-02  (0 s)
K5               n =  5  qubits = 5  rho = nan  tau = nan  max rel err = 3.8e-04  (0 s)
Sedgewick-Maze   n =  8  qubits = 5  rho = 1.000  tau = 1.000  max rel err = 5.1e-04  (0 s)
Florentine       n = 15  qubits = 6  rho = 1.000  tau = 1.000  max rel err = 2.8e-04  (1 s)
Lollipop L10,20  n = 30  qubits = 7  rho = 1.000  tau = 1.000  max rel err = 1.1e-09  (5 s)
Davis women      n = 32  qubits = 7  rho = 1.000  tau = 1.000  max rel err = 2.6e-04  (2 s)
Karate club      n = 34  qubits = 8  rho = 1.000  tau = 1.000  max rel err = 5.5e-03  (10 s)
Tutte            n = 46  qubits = 8  rho = 1.000  tau = 1.000  max rel err = 4.0e-06  (10 s)
BA(50,5,7)       n = 50  qubits = 8  rho = 1.000  tau = 1.000  max rel err = 7.2e-05  (32 s)


In [10]:
#this cell puts the results of all graphs into one table.
#columns of the results table (one row per graph, ideal circuits, beta = 1):
#n, edges        : number of vertices and edges of the graph
#qubits          : circuit size = register qubits (log2 of the padded size) + 1 block-encoding qubit + 1 real-part qubit
#bipartite       : True if the graph has no odd cycles; then the sinh (odd) part is exactly zero and the odd circuit is skipped
#classes         : number of distinct SC values (vertices with identical SC form one class)
#deg_even/deg_odd: degree of the Chebyshev polynomial for the cosh (even) and sinh (odd) part; the circuit makes one block-encoding call per degree
#pe_min/po_min   : smallest success probability of the even/odd circuit over the vertices (small = many shots needed); po_min is nan for bipartite graphs
#nx_diff         : largest difference between numpy SC and networkx subgraph_centrality (checks the classical reference)
#MAE             : mean absolute error between QSVT SC and exact SC over the vertices
#max_rel_err     : largest relative error |QSVT - exact| / exact over the vertices
#rho, tau        : Spearman and Kendall rank correlation between exact and QSVT SC (1 = identical ranking; exact ties are treated as one class); nan if all SC are equal (K5)
#topk, overlap   : fraction of the exact top-k vertices that QSVT also puts in its top-k (k = 3 for n <= 8, else 5)
tab = pd.DataFrame(rows).T
tab = tab[["n", "edges", "qubits", "bipartite", "classes", "deg_even", "deg_odd", "pe_min", "po_min", "nx_diff", "MAE", "max_rel_err", "rho", "tau", "topk", "overlap"]]
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print(tab.to_string(formatters={"pe_min": "{:.1e}".format, "po_min": "{:.1e}".format, "nx_diff": "{:.1e}".format,
                                "MAE": "{:.1e}".format, "max_rel_err": "{:.1e}".format, "rho": "{:.3f}".format,
                                "tau": "{:.3f}".format, "overlap": "{:.2f}".format}))

                  n edges qubits bipartite classes deg_even deg_odd   pe_min   po_min  nx_diff      MAE max_rel_err    rho    tau topk overlap
Bull              5     5      5     False       3        4       3  7.9e-02  4.7e-34  2.8e-14  1.4e-02     1.1e-02  1.000  1.000    3    1.00
K3,5              8    15      5      True       2        4    None  5.7e-02      NaN  3.2e-12  1.1e-01     2.4e-02  1.000  1.000    3    1.00
K5                5    10      5     False       1        6       7  6.0e-02  2.7e-02  5.6e-12  4.2e-03     3.8e-04    NaN    NaN    3     NaN
Sedgewick-Maze    8    10      5     False       8        6       7  1.9e-02  1.9e-08  1.4e-13  6.8e-04     5.1e-04  1.000  1.000    3    1.00
Florentine       15    20      6     False      15        8       7  5.7e-03  8.3e-10  5.0e-13  3.5e-04     2.8e-04  1.000  1.000    5    1.00
Lollipop L10,20  30    65      7     False      13       24      23  1.2e-07  2.1e-37  3.1e-11  8.0e-10     1.1e-09  1.000  1.000    5    1.00